# TradePredict AI - Model Training

This notebook demonstrates training the ML model for price prediction.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
%matplotlib inline

## 1. Fetch Training Data

In [ ]:
from app.services.data_fetcher import DataFetcher

fetcher = DataFetcher()

# Train on BTC/USDT
df = fetcher.fetch_all('BTC/USDT')
print(f"Fetched {len(df)} data points")
df.tail()

## 2. Train the Model

In [ ]:
from app.models.trainer import ModelTrainer

trainer = ModelTrainer()
result = trainer.train(df, epochs=50)
print(result)

## 3. Evaluate Model

In [ ]:
# Make predictions on training data
predictions = []
actuals = []

for i in range(60, len(df)):
    window = df.iloc[:i+1]
    pred = trainer.predict(window)
    if pred is not None:
        predictions.append(pred)
        actuals.append(window.iloc[-1]['close'])

predictions = np.array(predictions)
actuals = np.array(actuals)

metrics = trainer.evaluate(df, predictions, actuals[:len(predictions)])
print("Model Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

## 4. Train on Other Symbols

In [ ]:
# Train on ETH
df_eth = fetcher.fetch_all('ETH/USDT')
result_eth = trainer.train(df_eth, epochs=50)
print("ETH:", result_eth)

# Train on SOL
df_sol = fetcher.fetch_all('SOL/USDT')
result_sol = trainer.train(df_sol, epochs=50)
print("SOL:", result_sol)

## 5. Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(actuals, label='Actual', alpha=0.7)
ax.plot(predictions, label='Predicted', alpha=0.7)
ax.set_xlabel('Time')
ax.set_ylabel('Price')
ax.set_title('BTC/USDT - Actual vs Predicted')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Save Model

The model is automatically saved to `data/models/mlp_model.pkl`